# OPSD — Paper-Comparable Qwen3-1.7B Training on Google Colab

This notebook runs a **single-GPU** version of [OPSD (On-Policy Self-Distillation)](https://arxiv.org/pdf/2601.18734v3) configured for the paper-comparable Qwen3-1.7B iteration: 100 steps, checkpoints every 25 steps, student non-thinking rollouts, and teacher thinking/privileged scoring.

The full project targets **4×H100** with vLLM, FlashAttention-2, and DeepSpeed ZeRO-2. This notebook keeps the paper's loss/teacher/LoRA/sampling settings and scales only the compute:

| Setting | Full repo | This notebook |
|---|---|---|
| Model | Qwen3-1.7B / 4B / 8B | **Qwen3-1.7B** |
| GPUs | 4–8 | **1 (A100 40GB recommended)** |
| Generation backend | vLLM (colocate) | **vLLM (colocate)** |
| Attention (training) | `flash_attention_2` | **`sdpa`** |
| Launcher | `accelerate` + DeepSpeed | **plain `python`** |
| Rollout length | 1024 | **1024** |
| Effective batch | 32 | **16** |
| Steps | ~100 (peaks) | **100** |
| Checkpoints | 25/50/75/100 | **25/50/75/100** |

> **Before running:** set the runtime to a GPU via **Runtime → Change runtime type → Hardware accelerator: A100 GPU**.

This is now a paper-comparable training iteration, not a quick smoke test. On one GPU it may take longer than the original 4×H100 setup and can require memory tuning.

## 1. Check the GPU

Confirms a GPU is attached and detects whether it supports `bfloat16` (Ampere+ like A100/L4) or only `float16` (Turing T4).

In [1]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type and select a GPU."
)

gpu_name = torch.cuda.get_device_name(0)
BF16_OK = torch.cuda.is_bf16_supported()
print(f"\nGPU: {gpu_name}")
print(f"bfloat16 supported: {BF16_OK} (otherwise we fall back to float16)")

# Mount Google Drive so checkpoints/outputs persist across runtime resets.
from google.colab import drive
drive.mount("/content/drive")

Tue Jun 16 04:48:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   25C    P0             44W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Get the project code into Colab

The next cell sets `PROJECT_DIR` (the folder containing `opsd_train.py`) automatically:

1. **Google Drive (default).** If `DRIVE_PROJECT_DIR` (default `/content/drive/MyDrive/OPSD-main`) exists, it's used directly — no upload needed on reruns. Just put the `OPSD-main` folder in your Drive once.
2. **Already on `/content`.** If you extracted/cloned it earlier this session, it's auto-detected.
3. **Zip upload (fallback).** If neither is found, you'll be prompted to upload a `.zip` of the project.

**Alternative — Clone from GitHub.** If you've pushed the repo, use the commented clone cell below instead.

In [3]:
# Get the project code. Tries, in order:
#   1. Your Drive folder (DRIVE_PROJECT_DIR) — no upload needed on reruns.
#   2. An already-extracted copy anywhere under /content.
#   3. Zip upload as a last resort.
import os, glob, zipfile

# Adjust this if your OPSD folder lives elsewhere on Drive.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/OPSD-main"

def find_project_dir(root="/content"):
    hits = glob.glob(os.path.join(root, "**", "opsd_train.py"), recursive=True)
    return os.path.dirname(hits[0]) if hits else None

if os.path.exists(os.path.join(DRIVE_PROJECT_DIR, "opsd_train.py")):
    PROJECT_DIR = DRIVE_PROJECT_DIR
    print("Using project from Drive.")
else:
    PROJECT_DIR = find_project_dir()

if PROJECT_DIR is None:
    from google.colab import files
    print("Not found on Drive or /content. Upload a .zip of the OPSD project "
          "(the folder containing opsd_train.py)...")
    uploaded = files.upload()
    zip_name = next(n for n in uploaded if n.lower().endswith(".zip"))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("/content")
    PROJECT_DIR = find_project_dir()

assert PROJECT_DIR, "Could not locate opsd_train.py. Make sure it's on Drive or in the zip."
print("PROJECT_DIR =", PROJECT_DIR)
print(sorted(os.listdir(PROJECT_DIR)))

Using project from Drive.
PROJECT_DIR = /content/drive/MyDrive/OPSD-main
['.git', '.gitignore', 'README.md', '__pycache__', 'accelerate.yaml', 'colab_lightweight_eval.ipynb', 'colab_lightweight_train copy.ipynb', 'data_collator.py', 'environment.yml', 'eval', 'grpo_train.py', 'opsd_train.py', 'opsd_trainer.py', 'scripts', 'sft_train.py']


In [ ]:
# Alternative: clone from GitHub instead of Drive/zip (uncomment + set your URL).
# Drive is already mounted in the GPU cell, so you normally don't need this.
# REPO_URL = "https://github.com/<user>/<repo>.git"
# !git clone --depth 1 $REPO_URL /content/opsd_repo
# import glob, os
# PROJECT_DIR = os.path.dirname(glob.glob('/content/opsd_repo/**/opsd_train.py', recursive=True)[0])
# print('PROJECT_DIR =', PROJECT_DIR)

## 3. Install dependencies

We pin the TRL/Transformers stack to the versions this code was written against, and **add `vllm==0.11.0`** for fast on-policy rollouts (colocate mode) — this is what makes the paper recipe (1024-token generations, ~50 steps) affordable on a single A100 in ~30 min. We still skip `flash-attn`/`deepspeed` (training uses `sdpa`; vLLM handles its own attention).

> The vLLM install is heavier and may take a few minutes. If Colab prompts to **restart the runtime** after install, do it, then re-run from cell 1.

In [3]:
%pip install -q \
    "transformers==4.57.1" \
    "trl==0.26.0" \
    "peft==0.17.1" \
    "accelerate==1.11.0" \
    "datasets==3.6.0" \
    "math-verify==0.8.0" \
    "einops==0.8.1"

# vLLM gives fast on-policy rollouts (colocate mode) — needed to fit the paper
# recipe (1024-token generations, ~50 steps) into ~30 min on a single A100.
# It is installed AFTER the base stack, then transformers/trl are re-pinned so
# vLLM's dependency resolution can't silently downgrade them.
%pip install -q "vllm==0.11.0"

# vLLM 0.11.0 pins torch==2.8.0, but Colab's in-place upgrade can leave a mix of
# torch files from two versions, which surfaces as internal import errors like
# `ImportError: cannot import name 'CUSTOM_KEY' from 'torch.ao.quantization'`.
# Force a clean, single-version reinstall of torch so every torch file matches.
%pip install -q --force-reinstall --no-deps "torch==2.8.0"

%pip install -q "transformers==4.57.1" "trl==0.26.0"

# torchvision (Colab's preinstalled build) no longer matches torch 2.8.0 and breaks
# `import transformers` (`operator torchvision::nms does not exist`). This text-only
# pipeline never uses torchvision, so remove it; transformers skips the import when absent.
%pip uninstall -y -q torchvision

print("\nDone. >>> RESTART THE RUNTIME NOW (Runtime > Restart session) <<<, then re-run from cell 1.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 184.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.2/517.2 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# Sanity check that the trainer imports without vLLM / flash-attn / deepspeed
import sys
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import importlib
import opsd_trainer  # noqa: F401
importlib.reload(opsd_trainer)
print("opsd_trainer imported OK")

/content/drive/MyDrive/OPSD-main/opsd_trainer.py:63: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.gold.gold_config import GOLDConfig


opsd_trainer imported OK


## 4. Configure the paper-comparable run

These settings target the Qwen3-1.7B paper-comparable run: 100 steps, `MAX_COMPLETION=1024`, `MAX_LEN=20000`, LoRA rank 64, and checkpoints at 25/50/75/100. On a single A100, effective batch is kept at 16 for memory safety (the paper uses 32 across 4 GPUs).

In [5]:
# Paper-comparable Qwen3-1.7B main OPSD recipe.
# This matches scripts/run_opsd_1b.sh conceptually: student non-thinking rollout,
# teacher thinking/privileged scoring, 1024 rollout tokens, checkpoints at 25/50/75/100.
MODEL          = "Qwen/Qwen3-1.7B"
MAX_STEPS      = 100                   # paper reports checkpoints through step 100
MAX_COMPLETION = 1024                  # paper rollout length for distillation
MAX_LEN        = 20000                 # paper training context budget
OUTPUT_DIR     = "/content/drive/MyDrive/opsd_outputs"  # on Drive so checkpoints persist
RUN_NAME       = "colab_a100_thinking_paper"

# Effective batch = PER_DEVICE_BSZ * GRAD_ACCUM on this single GPU.
# Paper uses effective batch 32 over 4 GPUs; this single-GPU setting keeps memory safer.
PER_DEVICE_BSZ = 1
GRAD_ACCUM     = 16

# Fraction of GPU memory reserved for the colocated vLLM engine. The rest is left for
# training (model + activations + full-vocab logits). Lower this if you hit OOM.
VLLM_UTIL      = 0.45

DTYPE = "bfloat16" if BF16_OK else "float16"
# In transformers 4.57, --bf16/--fp16 require an explicit value (e.g. `--bf16 True`),
# not a bare flag, so we pass `True` here.
PREC_FLAG = "--bf16 True" if BF16_OK else "--fp16 True"
print(f"Model={MODEL}  dtype={DTYPE} ({PREC_FLAG})  eff_batch={PER_DEVICE_BSZ * GRAD_ACCUM}  steps={MAX_STEPS}  run={RUN_NAME}")

Model=Qwen/Qwen3-1.7B  dtype=bfloat16 (--bf16 True)  eff_batch=16  steps=100  run=colab_a100_thinking_paper


## 5. Run training

Runs `opsd_train.py` as a plain single-GPU process using the paper-comparable **Qwen3-1.7B main OPSD** setup:
- `--use_peft --fixed_teacher` with `--lora_r 64 --lora_alpha 128`: LoRA student with a frozen base-model teacher.
- `--use_vllm --vllm_mode colocate`: fast on-policy rollouts in the same process.
- `--beta 0 --lmbda 1 --jsd_token_clip 0.05`, `--temperature 1.1 --top_p 0.95 --top_k 20`: main paper loss + sampling recipe.
- `--student_thinking False --teacher_thinking True`: student samples in non-thinking mode; teacher scores with thinking/privileged context (the main OPSD style discussed in the paper/blog).
- `--max_steps 100 --save_steps 25`: saves checkpoints at 25/50/75/100 so the eval notebook can reproduce the doc-style per-step table.
- `--learning_rate 5e-6 --max_grad_norm 0.1`, effective batch 16 on one GPU (paper uses effective batch 32 over 4 GPUs), `--gradient_checkpointing` for memory.
- WandB disabled.

**Expectations:** this is now a paper-comparable iteration, not a 30-minute smoke test. On a single A100 it may take longer and can still OOM; if so, lower `MAX_LEN`, `VLLM_UTIL`, or `PER_DEVICE_BSZ`, but doing so moves away from the paper setting.

In [6]:
import subprocess, shlex, sys

cmd = f"""{sys.executable} -u opsd_train.py \
  --model_name_or_path {MODEL} \
  --attn_implementation sdpa \
  --torch_dtype {DTYPE} {PREC_FLAG} \
  --learning_rate 5e-6 \
  --max_grad_norm 0.1 \
  --per_device_train_batch_size {PER_DEVICE_BSZ} \
  --gradient_accumulation_steps {GRAD_ACCUM} \
  --gradient_checkpointing \
  --output_dir {OUTPUT_DIR} \
  --run_config {RUN_NAME} \
  --max_steps {MAX_STEPS} \
  --num_train_epochs 30 \
  --max_completion_length {MAX_COMPLETION} \
  --max_length {MAX_LEN} \
  --save_steps 25 \
  --logging_steps 1 \
  --beta 0 --lmbda 1 \
  --temperature 1.1 --top_p 0.95 --top_k 20 \
  --use_vllm --vllm_mode colocate --vllm_gpu_memory_utilization {VLLM_UTIL} --vllm_tensor_parallel_size 1 \
  --use_peft --lora_r 64 --lora_alpha 128 \
  --lora_target_modules q_proj k_proj v_proj o_proj gate_proj up_proj down_proj \
  --fixed_teacher \
  --student_thinking False --teacher_thinking True \
  --jsd_token_clip 0.05 \
  --report_to none"""

env = dict(os.environ)
env.update({
    "WANDB_MODE": "disabled",
    "WANDB_DISABLED": "true",
    "TOKENIZERS_PARALLELISM": "false",
    # Reduce CUDA memory fragmentation (recommended in the OOM traceback).
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})

print(cmd, "\n\n--- launching ---\n")
# Stream output live AND capture it so errors are visible (subprocess.run with no
# capture can swallow the traceback in Colab, leaving only "Exit code: 2").
proc = subprocess.Popen(
    shlex.split(cmd),
    cwd=PROJECT_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
output_lines = []
for line in proc.stdout:
    print(line, end="")
    output_lines.append(line)
proc.wait()
print("\nExit code:", proc.returncode)

Streaming output truncated to the last 5000 lines.
vLLM generation done - elapsed time: 4.17s, prompts: 1, total tokens: 1024, avg length: 1024.0, speed: 245.3 tok/s
vLLM generation done - elapsed time: 4.17s, prompts: 1, total tokens: 1024, avg length: 1024.0, speed: 245.4 tok/s
vLLM generation done - elapsed time: 1.52s, prompts: 1, total tokens: 373, avg length: 373.0, speed: 245.6 tok/s

Saved 96 generation outputs to:
  /content/drive/MyDrive/opsd_outputs/colab_a100_thinking_paper/generations/generations_step_5.json


  6%|▌         | 6/100 [05:49<1:29:15, 56.97s/it]
                                                 
{'loss': 0.003, 'grad_norm': 0.11298011243343353, 'learning_rate': 4.75e-06, 'on_policy_loss': 0.003, 'epoch': 0.0}

  6%|▌         | 6/100 [05:49<1:29:15, 56.97s/it]vLLM generation done - elapsed time: 4.17s, prompts: 1, total tokens: 1024, avg length: 1024.0, speed: 245.6 tok/s
vLLM generation done - elapsed time: 4.17s, prompts: 1, total tokens: 1024, avg length: 10

## 6. Inspect outputs & quick generation test

The trained LoRA adapter is saved under `OUTPUT_DIR/RUN_NAME`. Below we load the base model + adapter and generate a short answer to a math prompt to confirm the checkpoint is usable.

In [7]:
import os
adapter_dir = os.path.join(OUTPUT_DIR, RUN_NAME)
print("Adapter dir:", adapter_dir)
print(sorted(os.listdir(adapter_dir)) if os.path.isdir(adapter_dir) else "(not found)")

Adapter dir: /content/drive/MyDrive/opsd_outputs/colab_a100_thinking_paper
['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'added_tokens.json', 'chat_template.jinja', 'checkpoint-100', 'checkpoint-25', 'checkpoint-50', 'checkpoint-75', 'generations', 'merges.txt', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.json']


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

dtype = torch.bfloat16 if BF16_OK else torch.float16
tok = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype, attn_implementation="sdpa").cuda()
model = PeftModel.from_pretrained(base, adapter_dir).cuda().eval()

messages = [{"role": "user", "content": "What is 17 * 24? Show your reasoning briefly."}]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tok(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


To find $ 17 \times 24 $, we can use the distributive property:

$$
17 \times 24 = 17 \times (20 + 4) = (17 \times 20) + (17 \times 4)
$$

Now calculate each part:

$$
17 \times 20 = 340
$$
$$
17 \times 4 = 68
$$

Add the results:

$$
340 + 68 = 408
$$

### Final Answer:
$$
\boxed{408}
$$


## Next steps

- **After training:** open `colab_lightweight_eval.ipynb`; it is now configured for full AIME24 thinking-mode `Avg@12` and `INCLUDE_CHECKPOINTS=True`, so it will evaluate base plus checkpoints 25/50/75/100.
- **Runtime:** full paper-style eval is a separate expensive job (roughly 30-50 min per checkpoint on strong GPUs, longer on T4).
- **If training OOMs:** lower `VLLM_UTIL`, `MAX_LEN`, or `PER_DEVICE_BSZ`; this helps memory but moves away from the paper setting.
- **Multi-GPU / DeepSpeed:** for closer runtime/throughput parity, use the original `scripts/run_opsd_1b.sh` with `accelerate launch --config_file accelerate.yaml` on 4xH100.
- **Persisting results:** checkpoints save to Drive under `OUTPUT_DIR/RUN_NAME`, so they survive runtime resets.